# Reconnaissance faciale avec YOLOv8

Les modèles de deep learning apprennent des représentations hiérarchiques à partir des données brutes. Les **CNN** appliquent des filtres appris spatialement — détectant les contours, puis les textures, puis les structures de haut niveau. ResNet, par exemple, empile des blocs CNN résiduels pour classifier les images de manière fiable. Les **EfficientNets** font évoluer les CNN en profondeur, en largeur et en résolution via un coefficient composé, atteignant une meilleure précision par paramètre — EfficientNet-B0 égale ResNet-50 avec environ 5× moins de paramètres.

Ici, nous affinons **YOLOv8**, qui repose sur un backbone CNN, pour résoudre une tâche de classification binaire : *owner* vs *not_owner*.

In [ ]:
!pip install ultralytics

## 1. Imports

In [ ]:
import os
import cv2
import uuid
import shutil
import random
from ultralytics import YOLO

## 2. Collecte des images

Un classifieur a besoin d'exemples étiquetés. Nous capturons deux classes :
- **owner** — des photos de vous
- **not_owner** — des photos de n'importe qui d'autre, ou des arrière-plans variés

YOLOv8 classification attend la structure de dossiers suivante :

```
data/dataset/
├── train/
│   ├── owner/
│   └── not_owner/
└── val/
    ├── owner/
    └── not_owner/
```

Nous collectons d'abord les images brutes, puis les divisons automatiquement. Visez au minimum 40 à 50 images par classe.

Appuyez sur **S** pour sauvegarder une image, **Q** pour quitter.

In [ ]:
RAW_PATH = os.path.join('data', 'raw')
CLASSES = ['owner', 'not_owner']
NUM_IMAGES = 40  # par classe

for cls in CLASSES:
    os.makedirs(os.path.join(RAW_PATH, cls), exist_ok=True)

print("Dossiers prêts :", os.listdir(RAW_PATH))

In [ ]:
# Capture des images OWNER — placez-vous face à la caméra
cap = cv2.VideoCapture(0)
cls = 'owner'
count = 0

while cap.isOpened() and count < NUM_IMAGES:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()
    cv2.putText(display, f"{cls}  {count}/{NUM_IMAGES}  [S] sauvegarder  [Q] quitter",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    cv2.imshow('Capture', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        path = os.path.join(RAW_PATH, cls, f"{cls}_{uuid.uuid1()}.jpg")
        cv2.imwrite(path, frame)
        count += 1
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"{count} images sauvegardées pour '{cls}'")

In [ ]:
# Capture des images NOT_OWNER — demandez à quelqu'un d'autre de se placer devant la caméra, ou éloignez-vous
cap = cv2.VideoCapture(0)
cls = 'not_owner'
count = 0

while cap.isOpened() and count < NUM_IMAGES:
    ret, frame = cap.read()
    if not ret:
        break

    display = frame.copy()
    cv2.putText(display, f"{cls}  {count}/{NUM_IMAGES}  [S] sauvegarder  [Q] quitter",
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    cv2.imshow('Capture', display)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s'):
        path = os.path.join(RAW_PATH, cls, f"{cls}_{uuid.uuid1()}.jpg")
        cv2.imwrite(path, frame)
        count += 1
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"{count} images sauvegardées pour '{cls}'")

## 3. Construction du jeu de données

Nous divisons chaque classe 80/20 entre entraînement et validation. Le modèle apprend sur la partition d'entraînement ; la partition de validation mesure sa capacité à généraliser sur des images qu'il n'a jamais vues — ce qui compte en pratique.

In [ ]:
DATASET_PATH = os.path.join('data', 'dataset')
VAL_SPLIT = 0.2

for cls in CLASSES:
    src = os.path.join(RAW_PATH, cls)
    images = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(images)

    cut = int(len(images) * (1 - VAL_SPLIT))
    splits = {'train': images[:cut], 'val': images[cut:]}

    for split, files in splits.items():
        dest = os.path.join(DATASET_PATH, split, cls)
        os.makedirs(dest, exist_ok=True)
        for f in files:
            shutil.copy(os.path.join(src, f), os.path.join(dest, f))

    print(f"{cls}: {cut} train / {len(images) - cut} val")

print("\nDataset ready at:", DATASET_PATH)

## 4. Affinage de YOLOv8

Plutôt qu'entraîner depuis zéro, nous chargeons `yolov8n-cls.pt` — un modèle nano pré-entraîné sur ImageNet. Nous remplaçons et ré-entraînons uniquement sa tête de classification sur nos deux classes. C'est le **transfer learning** : le backbone sait déjà extraire des caractéristiques visuelles ; il suffit de lui apprendre à reconnaître votre visage spécifique. La convergence est rapide et fonctionne bien même avec peu d'images.

In [ ]:
model = YOLO('yolov8n-cls.pt')

model.train(
    data=DATASET_PATH,
    epochs=30,
    imgsz=224,
    batch=16,
    name='face_recognition',
    patience=10,   # arrêt anticipé si la perte de validation ne s'améliore plus
)

print("Meilleurs poids sauvegardés dans :", model.trainer.best)

## 5. Test sur une image fixe

Avant de brancher le modèle dans l'application en direct, vérifiez qu'il prédit correctement sur un échantillon de validation.

In [ ]:
best_model = YOLO(os.path.join('runs', 'classify', 'face_recognition', 'weights', 'best.pt'))

val_owner_dir = os.path.join(DATASET_PATH, 'val', 'owner')
sample = os.path.join(val_owner_dir, os.listdir(val_owner_dir)[0])

results = best_model(sample)
probs = results[0].probs
print("Classe prédite :", results[0].names[probs.top1])
print(f"Confiance : {probs.top1conf:.2%}")

## 6. Test en direct avec la webcam

Une dernière vérification avant le déploiement : faites tourner le modèle en temps réel sur le flux de la webcam. Vert = owner, rouge = not_owner. Appuyez sur **Q** pour quitter.

In [ ]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = best_model(frame, verbose=False)
    probs = results[0].probs
    label = results[0].names[probs.top1]
    conf = probs.top1conf.item()

    color = (0, 200, 80) if label == 'owner' else (0, 60, 220)
    cv2.putText(frame, f"{label}  {conf:.0%}", (12, 42),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2)
    cv2.imshow('Face Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
cap = cv2.VideoCapture(0)
# Loop through labels
for label in labels:
    print('Collecting images for {}'.format(label))
    time.sleep(5)
    
    # Loop through image range
    for img_num in range(number_imgs):
        print('Collecting images for {}, image number {}'.format(label, img_num))
        
        # Webcam feed
        ret, frame = cap.read()
        
        # Naming out image path
        imgname = os.path.join(IMAGES_PATH, label+'.'+str(uuid.uuid1())+'.jpg')
        
        # Writes out image to file 
        cv2.imwrite(imgname, frame)
        
        # Render to the screen
        cv2.imshow('Image Collection', frame)
        
        # 2 second delay between captures
        time.sleep(2)
        
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
cap.release()
cv2.destroyAllWindows()

In [ ]:
for label in labels:
    print('Collecting images for {}'.format(label))
    for img_num in range(number_imgs):
        print('Collecting images for {}, image number {}'.format(label, img_num))
        imgname = os.path.join(IMAGES_PATH, label+'.'+str(uuid.uuid1())+'.jpg')
        print(imgname)   

In [ ]:
!pip install pyqt5 lxml --upgrade
!cd labelImg && pyrcc5 -o libs/resources.py resources.qrc

# 6. Load Custom Model

In [11]:
img = os.path.join('data', 'images', 'awake.c9a24d48-e1f6-11eb-bbef-5cf3709bbcc6.jpg')

In [13]:
results.print()

image 1/1: 480x640 1 awake
Speed: 16.0ms pre-process, 11.0ms inference, 2.0ms NMS per image at shape (1, 3, 480, 640)


In [15]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    
    # Make detections 
    results = model(frame)
    
    cv2.imshow('YOLO', np.squeeze(results.render()))
    
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()